# Productionize product_matcher
Generated 2026-05-09 from `models/product_matcher/`.

- task: classification
- estimator: `HistGradientBoostingClassifier` from `sklearn.ensemble._hist_gradient_boosting.gradient_boosting`
- metric: validation_auc=0.9974 on 1000 test rows
- bundle saved: 2026-05-09T16:59:02Z (sklearn 1.3.2, python 3.12.9)
- data: `data/matcher/training.parquet`, 5000 rows total (4000 train / 1000 test)

This notebook is a *starting point* — edit freely. The reconstruct-from-config cell is the production-ready path: it rebuilds the estimator from `config.json` without touching the pickle, proving the bundle's weights/code split.

In [ ]:
import os, json
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
%load_ext autoreload
%autoreload 2

## Inspect the bundle

In [ ]:
config = json.loads(open('models/product_matcher/config.json').read())
metadata = json.loads(open('models/product_matcher/metadata.json').read())
print(json.dumps(config, indent=2)[:500])
print('---')
print(json.dumps(metadata, indent=2)[:500])

## Reload the data

The lineage records this dataset was generated by `prepare_matcher_pairs` from the synthetic `product_catalog` scenario (seed=42, n_pairs=5000, balanced positives). To regenerate from scratch: `python prep/prepare_matcher_pairs.py` (or `make data-matcher` if wired).

In [ ]:
import pandas as pd
df = pd.read_parquet('data/matcher/training.parquet')
print(df.shape)
df.describe().T.head(12)

## Reconstruct the model from config (no pickle)

In [ ]:
import importlib
mod = importlib.import_module(config['estimator_module'])
EstimatorClass = getattr(mod, config['estimator'])
model_from_config = EstimatorClass(**config['params'])
model_from_config

## Reproduce the train/test split and refit

In [ ]:
from plugins import get_plugin
from sklearn.model_selection import train_test_split

plugin = get_plugin(config['plugin'])
X, y = plugin.prepare(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model_from_config.fit(X_train, y_train)
print(f'fit OK — {len(X_train)} train / {len(X_test)} test rows')

## Sanity-check vs the bundled weights

Load the joblib pickle and confirm probabilities match. This is what proves the config-only rebuild is faithful.

In [ ]:
import joblib, numpy as np
model_from_bundle = joblib.load('models/product_matcher/model.joblib')

np.testing.assert_allclose(
    model_from_config.predict_proba(X_test[:50]),
    model_from_bundle.predict_proba(X_test[:50]),
    rtol=1e-5,
)
print('OK — config-rebuilt model matches bundled weights')

## Re-evaluate

In [ ]:
import inspect
n_params = len(inspect.signature(plugin.evaluate).parameters)
if n_params >= 3:
    name, value = plugin.evaluate(model_from_config, X_test, y_test)
else:
    name, value = plugin.evaluate(y_test, model_from_config.predict(X_test))
bundle_metric = metadata[config['metric_name']]
print(f'{name}={value:.4f}  (bundle reported {config["metric_name"]}={bundle_metric:.4f})')

## EDA — target balance and confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

y.value_counts().sort_index().plot.bar(ax=axes[0], color=['#888', '#2a9d8f'])
axes[0].set_title('Target distribution')
axes[0].set_xlabel('same_canonical')
axes[0].set_ylabel('count')

ConfusionMatrixDisplay.from_estimator(model_from_config, X_test, y_test, ax=axes[1])
axes[1].set_title('Confusion matrix (test)')

RocCurveDisplay.from_estimator(model_from_config, X_test, y_test, ax=axes[2])
axes[2].set_title('ROC')

plt.tight_layout(); plt.show()

## Feature importance (permutation)

`HistGradientBoostingClassifier` doesn't expose `feature_importances_`; use permutation importance. Watch for the GTIN-as-feature trap: a healthy model leans on title similarity in addition to `same_gtin`.

In [ ]:
from sklearn.inspection import permutation_importance
import numpy as np

result = permutation_importance(
    model_from_config, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)
order = np.argsort(result.importances_mean)[::-1]
names = np.array(config['feature_names'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(names[order][::-1], result.importances_mean[order][::-1])
ax.set_xlabel('Permutation importance (Δ AUC)')
ax.set_title('Feature importance')
plt.tight_layout(); plt.show()

## Production checklist

- [ ] **Pin dependencies** — bundle was trained with `sklearn==1.3.2`, python `3.12.9`. Cross-version joblib loads are brittle; the config-rebuild path above sidesteps this — prefer it over unpickling in production.
- [ ] **Lock data lineage** — confirm `metadata.data_lineage.dataset_sha256` (`c4ee8a71…`) is reproducible from the same simulation seed (42). If you swap to a real candidate-pair source, parameterize the prep script.
- [ ] **Decide retraining cadence** — catalog churn (new SKUs, retailer onboarding) is the natural trigger; consider event-driven retrain on catalog deltas above some threshold.
- [ ] **Wire metric monitoring** — alert if `validation_auc` drops below ~0.95 on production traffic. Watch class balance drift too: production candidate pairs are likely far more skewed than the 50/50 sampled training set, so calibrate the threshold accordingly.
- [ ] **Tune `prediction_threshold`** — 0.5 is appropriate for balanced training but production pair generation is asymmetric (most candidate pairs are negatives); raise the threshold to reduce false positives.
- [ ] **Sanity-check the GTIN trap** — confirm permutation importance shows the model uses title similarity in addition to `same_gtin`. A model that has collapsed to GTIN-only will fail on eBay-style pairs without GTINs.
- [ ] **If deploying to SageMaker:** see `pipeline.py` (training pipeline) and `deploy_endpoint.py` (endpoint deploy). Ship `config.json` next to the weights so the inference container can rebuild via the same path used here.
- [ ] **Smoke-test against `local_serve.py`** before deploy — same sklearn version host vs container, otherwise expect silent prediction skew.
